In [1]:
# ============================================================
# Assignment 5.1 - SageMaker Training Notebook
# Airstay Insights - Airbnb Price Prediction
# ============================================================

import pandas as pd
import numpy as np
import boto3
import sagemaker
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve

# -----------------------------
# 1. SageMaker session setup
# -----------------------------
session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()

# Use your team bucket if you have one, otherwise default bucket is fine
bucket = session.default_bucket()
prefix = "airstay-insights/assignment-5-1"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Bucket: sagemaker-us-east-1-316001945332
Role: arn:aws:iam::316001945332:role/LabRole


In [2]:
# -----------------------------
# 2. Load data
# -----------------------------
# If the CSV is already in the notebook folder:
df = pd.read_csv("AB_NYC_2019.csv")

# If needed, you could also load from S3 directly:
# s3_path = f"s3://{bucket}/{prefix}/raw/AB_NYC_2019.csv"
# df = pd.read_csv(s3_path)

print(df.shape)
df.head()

(48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
# -----------------------------
# 3. Clean data
# -----------------------------
df = df.drop_duplicates()

df = df.drop(
    columns=["id", "name", "host_id", "host_name", "last_review"],
    errors="ignore"
)

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

df = df.dropna(subset=["neighbourhood_group", "neighbourhood", "room_type", "price"])

# Keep realistic prices
df = df[(df["price"] > 0) & (df["price"] <= 500)]

print(df.shape)
print(df.isnull().sum())

(47840, 11)
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64


In [4]:
# -----------------------------
# 4. Feature engineering
# -----------------------------
# Target
y = df["price"].copy()

# Features
X = df.drop(columns=["price"], errors="ignore")

# One-hot encode categorical columns
categorical_cols = ["neighbourhood_group", "neighbourhood", "room_type"]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Ensure all boolean columns become integers
for col in X.columns:
    if X[col].dtype == bool:
        X[col] = X[col].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (47840, 231)
Target shape: (47840,)


In [5]:
# -----------------------------
# 5. Train / validation / test split
# -----------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (30617, 231)
Validation shape: (7655, 231)
Test shape: (9568, 231)


In [6]:
# -----------------------------
# 6. Format for SageMaker XGBoost
# SageMaker built-in XGBoost expects target column first
# -----------------------------
train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([y_val.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
test_df  = pd.concat([y_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

train_file = "train.csv"
val_file = "validation.csv"
test_file = "test.csv"

train_df.to_csv(train_file, index=False, header=False)
val_df.to_csv(val_file, index=False, header=False)
test_df.to_csv(test_file, index=False, header=False)

In [7]:
# -----------------------------
# 7. Upload train/validation/test to S3
# -----------------------------
train_s3_uri = session.upload_data(path=train_file, bucket=bucket, key_prefix=f"{prefix}/train")
val_s3_uri = session.upload_data(path=val_file, bucket=bucket, key_prefix=f"{prefix}/validation")
test_s3_uri = session.upload_data(path=test_file, bucket=bucket, key_prefix=f"{prefix}/test")

print("Train S3 URI:", train_s3_uri)
print("Validation S3 URI:", val_s3_uri)
print("Test S3 URI:", test_s3_uri)

Train S3 URI: s3://sagemaker-us-east-1-316001945332/airstay-insights/assignment-5-1/train/train.csv
Validation S3 URI: s3://sagemaker-us-east-1-316001945332/airstay-insights/assignment-5-1/validation/validation.csv
Test S3 URI: s3://sagemaker-us-east-1-316001945332/airstay-insights/assignment-5-1/test/test.csv


In [8]:
# -----------------------------
# 8. Create SageMaker XGBoost estimator
# -----------------------------
container = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session
)

xgb.set_hyperparameters(
    objective="reg:squarederror",
    num_round=200,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="rmse"
)

In [9]:
# -----------------------------
# 9. Launch SageMaker training job
# -----------------------------
xgb.fit({
    "train": TrainingInput(train_s3_uri, content_type="text/csv"),
    "validation": TrainingInput(val_s3_uri, content_type="text/csv")
})

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-29-23-35-18-231


2026-03-29 23:35:21 Starting - Starting the training job...
2026-03-29 23:35:36 Starting - Preparing the instances for training...
2026-03-29 23:35:59 Downloading - Downloading input data...
2026-03-29 23:36:44 Downloading - Downloading the training image......
2026-03-29 23:37:45 Training - Training image download completed. Training in progress...../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-29 23:37:50.469 ip-10-2-104-56.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-29 23:37:50.533 ip-10-2-104-56.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-29:23:37:50:INFO] Imported framework sagemaker_xgboost_container.trai

In [10]:
# -----------------------------
# 10. Deploy model temporarily for prediction
# -----------------------------
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-29-23-41-27-640
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-03-29-23-41-27-640
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-03-29-23-41-27-640


------!

In [12]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Convert test features to numpy
X_test_for_pred = X_test.copy().values

# Predict in batches to avoid 413 payload-too-large error
batch_size = 500
all_predictions = []

for i in range(0, len(X_test_for_pred), batch_size):
    batch = X_test_for_pred[i:i+batch_size]
    batch_pred = predictor.predict(batch)
    batch_pred = np.array(batch_pred).astype(float).flatten()
    all_predictions.extend(batch_pred)

y_pred = np.array(all_predictions)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 61.103137521909645
MAE: 40.67869569964233
R2: 0.5166879635973345


In [13]:
# -----------------------------
# 12. Clean up endpoint so you do not keep getting charged
# -----------------------------
predictor.delete_endpoint()
predictor.delete_model()

INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-xgboost-2026-03-29-23-41-27-640
INFO:sagemaker:Deleting endpoint with name: sagemaker-xgboost-2026-03-29-23-41-27-640


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│   2 # 12. Clean up endpoint so you do not keep getting charged                                   │
│   3 # -----------------------------                                                              │
│   4 predictor.delete_endpoint()                                                                  │
│ ❱ 5 predictor.delete_model()                                                                     │
│   6                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:722 in delete_model          │
│                                                                                                  │
│   719 │   │   """Delete the Amazon SageMaker model backing this predictor."""                    │
│   720 │   │   request_failed = False                                                             │
│   721 │   │   failed_models = []                                                                 │
│ ❱ 722 │   │   current_model_names = self._get_model_names()                                      │
│   723 │   │   for model_name in current_model_names:                                             │
│   724 │   │   │   try:                                                                           │
│   725 │   │   │   │   self.sagemaker_session.delete_model(model_name)                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:924 in _get_model_names      │
│                                                                                                  │
│   921 │   │   │   return self._model_names                                                       │
│   922 │   │                                                                                      │
│   923 │   │   current_endpoint_config_name = self._get_endpoint_config_name()                    │
│ ❱ 924 │   │   endpoint_config = self.sagemaker_session.sagemaker_client.describe_endpoint_conf   │
│   925 │   │   │   EndpointConfigName=current_endpoint_config_name                                │
│   926 │   │   )                                                                                  │
│   927 │   │   production_variants = endpoint_config["ProductionVariants"]                        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py: